### ЗАДАЧА: Бронирование переговорок

Офис-менеджер получает пачку заявок на переговорки.
Нужно собрать систему, которая:
- принимает корректные бронирования,
- отбрасывает конфликтующие или неправильные заявки,
- хранит расписание по комнатам,
- позволяет быстро понять, какая переговорка загружена сильнее всего.

В некоторых заявках указана неизвестная комната,
в некоторых время перепутано,
а некоторые пересекаются с уже занятыми слотами.


In [ ]:
from dataclasses import dataclass
from typing import Optional


rooms = {'A-101', 'B-204', 'C-305'}
# rows: booking_id|room_id|owner|start_hour|end_hour
rows = [
    'BK-100|A-101|Alice|9|11',
    'BK-101|A-101|Bob|10|12',
    'BK-102|B-204|Kira|13|15',
    'BK-103|X-999|Oleg|11|12',
    'BK-104|C-305|Eva|16|15',
    'BK-105|B-204|Max|15|17',
]


class BookingError(Exception):
    pass


class RoomNotFoundError(BookingError):
    pass


class TimeRangeError(BookingError):
    pass


class TimeConflictError(BookingError):
    pass


@dataclass(order=True)
class Booking:
    start_hour: int
    end_hour: int
    booking_id: str
    room_id: str
    owner: str


class RoomSchedule:
    def __init__(self, room_id):
        # TODO: сохранить room_id
        # TODO: создать список bookings
        self.room_id = room_id
        self.bookings = []

    def can_add(self, booking):
        # TODO: пройтись по уже существующим booking в self.bookings
        # TODO: проверить пересечение интервалов
        # TODO: если пересечение есть -> вернуть False
        # TODO: если конфликтов нет -> вернуть True
        for el in self.bookings:
            if not (booking.end_hour <= el.start_hour or booking.start_hour >= el.end_hour):
                return False
        return True
           

    def add_booking(self, booking):
        # TODO: если can_add(...) == False -> raise TimeConflictError(...)
        # TODO: добавить booking в self.bookings
        # TODO: отсортировать self.bookings
        if self.can_add(booking) == False:
            raise TimeConflictError("Пересечение времени")
        self.bookings.append(booking)
        self.bookings.sort(key=lambda x: x.start_hour)
        

    def total_reserved_hours(self):
        # TODO: вернуть сумму (end_hour - start_hour) по всем бронированиям комнаты
        return sum(b.end_hour - b.start_hour for b in self.bookings)


class BookingService:
    def __init__(self, rooms):
        # TODO: создать schedules вида room_id -> RoomSchedule(room_id)
        self.schedules: dict[str, RoomSchedule] = {room: RoomSchedule(room) for room in rooms}
        # TODO: создать списки accepted и errors
        self.accepted = []
        self.errors = []
        

    def parse_booking(self, row):
        # TODO: split по '|'
        parts = row.split("|")
        if len(parts) != 5:
            raise BookingError("Неверный формат строки")
        # TODO: ожидать 5 частей: booking_id, room_id, owner, start_raw, end_raw
        booking_id, room_id, owner, start_str, end_str = parts
        # TODO: start_raw и end_raw преобразовать в int
        try:
            start_hour = int(start_str)
            end_hour = int(end_str)
        except ValueError:
            raise TimeRangeError("Время должно быть числом")
        # TODO: если room_id не существует -> RoomNotFoundError
        if room_id not in self.schedules:
            raise RoomNotFoundError ("Такой комнаты не существует")
        # TODO: если start_hour >= end_hour -> TimeRangeError
        if start_hour >= end_hour:
            raise TimeRangeError("Неправильное время")
        # TODO: вернуть объект Booking(...)
        return Booking(start_hour, end_hour, booking_id, room_id, owner)

    def submit(self, row):
        # TODO: внутри try вызвать parse_booking(row)
        # TODO: затем schedules[booking.room_id].add_booking(booking)
        # TODO: успех добавить в self.accepted
        # TODO: BookingError сохранить в self.errors как (row, error_type, message)
        try:
            booking = self.parse_booking(row)
            self.schedules[booking.room_id].add_booking(booking)
            self.accepted.append(booking)
        except BookingError as e:
            self.errors.append((row, type(e).__name__, str(e)))

    def load(self, rows):
        # TODO: вызвать submit(row) для каждой строки
        for row in rows:
            self.submit(row)

    def busiest_room(self):
        # TODO: найти комнату с максимумом total_reserved_hours()
        # TODO: вернуть tuple(room_id, total_hours)
        max_room_id = None
        max_hour = 0
        for room_id, rs in self.schedules.items():
            total_reserved_hours = rs.total_reserved_hours()
            if total_reserved_hours > max_hour:
                max_room_id = room_id
                max_hour = total_reserved_hours
        return f"Самая загруженная комната {max_room_id}: {max_hour} часов"


    def find_booking(self, booking_id) -> Optional[Booking]:
        # TODO: вернуть Optional[Booking]
        # TODO: пройтись по всем schedules и по всем bookings внутри них
        # TODO: если booking.booking_id совпал -> вернуть booking
        # TODO: если не найдено -> вернуть None
        for schedule in self.schedules.values():
            for booking in schedule.bookings:
                if booking.booking_id == booking_id:
                    return booking
        return None


service = BookingService(rooms)
service.load(rows)
print("Принятые бронирования:", len(service.accepted))
for el in service.accepted:
    print(el)

print("Ошибки:", len(service.errors))
for error in service.errors:
    print(error)

print(service.busiest_room())
print("Расписание BK-102:", service.find_booking('BK-102'))

# TODO: загрузить rows через service.load(rows)
# TODO: вывести принятые бронирования
# TODO: вывести ошибки
# TODO: вывести расписание по всем комнатам
# TODO: вывести busiest_room()
# TODO: вывести find_booking('BK-102')

Принятые бронирования: 3
Ошибки: 3
('BK-101|A-101|Bob|10|12', 'TimeConflictError', 'Пересечение времени')
('BK-103|X-999|Oleg|11|12', 'RoomNotFoundError', 'Такой комнаты не существует')
('BK-104|C-305|Eva|16|15', 'TimeRangeError', 'Неправильное время')
Booking(start_hour=9, end_hour=11, booking_id='BK-100', room_id='A-101', owner='Alice')
Booking(start_hour=13, end_hour=15, booking_id='BK-102', room_id='B-204', owner='Kira')
Booking(start_hour=15, end_hour=17, booking_id='BK-105', room_id='B-204', owner='Max')
Самая загруженная комната B-204: 4 часов
Расписание BK-102: Booking(start_hour=13, end_hour=15, booking_id='BK-102', room_id='B-204', owner='Kira')
